# SYNAPSE — Train & Publish the Demand Prophet ($0, free-GPU)

Reproducible **operator step** (ADR-043 / C43). Runs the *exact* production training
path in `agents/demand_prophet/training/train.py` (no divergent code), fits the
conformal calibrator, and publishes the checkpoint + serving sidecar to the **free**
Hugging Face Hub so the `$0` `ModelRegistry` serving source can resolve it.

**Run on:** Google Colab or Kaggle (free T4 GPU) — or any CPU box (slower).
**Cost:** $0 (I-1). **Output:** `demand_prophet_hgt_tft.pt` + `.serving.json` on HF Hub,
plus the registry line to paste into `infrastructure/ml/published_checkpoints.json`.

> After this notebook: set `DP_HF_REPO=<your-repo>` on the serving container and the
> C43 gate (`scripts/audit/published_checkpoint_truth.py`) flips SKIP → PASS.

## 1. Clone the repo + install the pinned dependency stack

In [ ]:
import os, subprocess, sys

REPO_URL = os.environ.get('SYNAPSE_REPO_URL', 'https://github.com/Aegis15/synapse.git')
BRANCH = os.environ.get('SYNAPSE_BRANCH', 'main')
if not os.path.isdir('synapse'):
    # Private repo: provide a token via the GITHUB_TOKEN secret (read-only).
    tok = os.environ.get('GITHUB_TOKEN')
    url = REPO_URL.replace('https://', f'https://x-access-token:{tok}@') if tok else REPO_URL
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', url, 'synapse'], check=True)
os.chdir('synapse')
print('cwd =', os.getcwd())

In [ ]:
# GPU torch on Colab/Kaggle is preinstalled; otherwise install CPU wheels. Then the
# pinned project + agent deps (all MIT/BSD/Apache — license-clean, I-1).
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                '-r', 'packages/requirements.txt',
                '-r', 'agents/demand_prophet/requirements.txt',
                'huggingface_hub'], check=True)
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Build the real feature store (1.1M-row Bengaluru series)

`build_supervised(city='bengaluru')` reads `data/bengaluru/demand_history.csv`. If your
checkout ships the parquet via `scripts/build_feature_store.py`, run it first.

In [ ]:
import pathlib
if pathlib.Path('scripts/build_feature_store.py').exists():
    subprocess.run([sys.executable, 'scripts/build_feature_store.py', '--city', 'bengaluru'], check=False)
print('demand_history present:', pathlib.Path('data/bengaluru/demand_history.csv').exists())

## 3. Full train (real gradient steps) + fit the conformal calibrator

Calls the production `train()` unchanged. It writes
`artifacts/checkpoints/demand_prophet_hgt_tft.pt` + `.serving.json` and returns a
`TrainResult` whose `assert_learned()` fails loudly if the loop did not actually learn.

In [ ]:
from agents.demand_prophet.config import DemandProphetConfig
from agents.demand_prophet.training.train import train, SERVING_NAME, CHECKPOINT_DIR

EPOCHS = int(os.environ.get('DP_EPOCHS', '50'))
config = DemandProphetConfig(max_epochs=EPOCHS, batch_size=64, learning_rate=1e-3)
result = train(config)            # full (non-smoke) production run
print('gradient_steps =', result.gradient_steps)
print('start_loss     =', result.start_loss)
print('end_loss       =', result.end_loss)
print('coverage_p90   =', result.metrics.get('coverage_p90'))
print('checkpoint_sha =', result.checkpoint_sha)

In [ ]:
# Hard gate before publishing: a production model must beat the INV-DP-002 floor.
cov = float(result.metrics.get('coverage_p90') or 0.0)
assert not result.smoke, 'refusing to publish a smoke artifact'
assert cov >= 0.85, f'coverage_p90={cov} < 0.85 floor — do not publish'
print('OK to publish: coverage', cov)

## 4. Publish checkpoint + sidecar to the free HF Hub

Set `HF_TOKEN` (a free write token from huggingface.co/settings/tokens) and `HF_REPO`
(e.g. `Praneshrajan15/synapse-demand-prophet`).

In [ ]:
from huggingface_hub import HfApi

HF_REPO = os.environ['HF_REPO']
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(HF_REPO, repo_type='model', exist_ok=True, private=False)
for fname in (f'{SERVING_NAME}.pt', f'{SERVING_NAME}.serving.json'):
    api.upload_file(path_or_fileobj=str(CHECKPOINT_DIR / fname),
                    path_in_repo=fname, repo_id=HF_REPO, repo_type='model')
    print('uploaded', fname)

## 5. Record the result (paste into `infrastructure/ml/published_checkpoints.json`)

Replace the `__placeholder__` object with the printed entry, commit it, and record the
same numbers in `docs/quality_gates/flagship-slice-evidence.md`. The **C43** gate then
verifies the published sidecar matches this record on every CI run.

In [ ]:
import json, datetime
entry = {SERVING_NAME: {
    'repo': HF_REPO,
    'sha': result.checkpoint_sha[:7],
    'coverage_p90': round(cov, 4),
    'final_crps': round(float(result.end_loss), 6),
    'trained_at': datetime.date.today().isoformat(),
}}
print(json.dumps(entry, indent=2, sort_keys=True))